# Predicting next-month arrears
Question: can we predict `arrears_next_month` from what is known at the end of the billing month?

Design decisions (from checklist.md and columns.csv):
- `meter_ref` is an identifier: dropped as a feature, used only for grouping.
- `final_notice_sent` is recorded in the cycle *after* the billing month, so it is not known at prediction time (and is likely a consequence of the outcome). Dropped.
- Features: `tariff`, `occupants`, `kwh`, `bill_eur`, `late_payments_12m`.
- Model is meant for future rows, so the test set is the latest months; accounts recur, so test accounts are held out of training entirely.

In [ ]:
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv('data.csv', parse_dates=['billing_month'])
TARGET='arrears_next_month'
FEATURES=['tariff','occupants','kwh','bill_eur','late_payments_12m']
print(df.shape, df.meter_ref.nunique(), 'accounts;', df.billing_month.min().date(), '->', df.billing_month.max().date())
df.groupby('billing_month')[TARGET].agg(['mean','size']).tail(14)

## 1. Test set, set aside before any modelling
Test = rows from the last 3 billing months (2025-10 onward) belonging to a random 25% of accounts. Training = rows before 2025-10 from the other accounts. So test rows are later than, and from different accounts than, every training row.

In [ ]:
rng=np.random.default_rng(0)
accts=df.meter_ref.unique()
test_accts=set(rng.choice(accts, size=len(accts)//4, replace=False))
CUTOFF=pd.Timestamp('2025-10-01')
is_test=df.meter_ref.isin(test_accts)&(df.billing_month>=CUTOFF)
is_train=~df.meter_ref.isin(test_accts)&(df.billing_month<CUTOFF)
test=df[is_test].copy(); train=df[is_train].sort_values('billing_month').reset_index(drop=True)
print(len(train), len(test), 'unused rows:', len(df)-len(train)-len(test))
assert not set(train.meter_ref)&set(test.meter_ref)
print('train prevalence %.3f' % train[TARGET].mean())

## 2. Candidates and plan (declared before scoring)
Candidates:
1. `prevalence` - trivial baseline, constant predicted probability.
2. `late_rule` - trivial rule baseline: score = `late_payments_12m`.
3. `logreg` - standardised numeric + one-hot tariff, logistic regression (C in {0.1, 1, 10}).
4. `hgb` - histogram gradient boosting (max_depth in {2, 3}, learning_rate 0.05, 200 iterations).

Selection metric: mean average precision across CV folds (outcome is imbalanced and prevalence drifts upward). Threshold: the one maximising F1 on pooled out-of-fold predictions of the chosen model. Nothing is chosen from the test set.

CV: forward-chaining on billing month inside the training rows; each validation fold is a later block of months, training folds contain only earlier months. (Accounts recur in every month, so also excluding validation accounts from earlier rows would leave nothing to train on; the checklist allows either temporal or whole-entity folds, and we use temporal.)

In [ ]:
def pre():
    return ColumnTransformer([('num',StandardScaler(),['occupants','kwh','bill_eur','late_payments_12m']),
                              ('cat',OneHotEncoder(handle_unknown='ignore'),['tariff'])])
class LateRule:
    def fit(self,X,y): return self
    def predict_proba(self,X):
        s=np.clip(X['late_payments_12m'].to_numpy(float)/10,0,1)
        return np.c_[1-s,s]
def candidates():
    c={'prevalence':DummyClassifier(strategy='prior'),'late_rule':LateRule()}
    for C_ in (0.1,1,10): c[f'logreg_C{C_}']=make_pipeline(pre(),LogisticRegression(C=C_,max_iter=1000))
    for d in (2,3): c[f'hgb_d{d}']=make_pipeline(pre(),HistGradientBoostingClassifier(max_depth=d,learning_rate=0.05,max_iter=200,random_state=0))
    return c
months=np.sort(train.billing_month.unique())
# 5 validation blocks of consecutive months at the end of the training period
blocks=np.array_split(months[-15:],5)
def folds():
    for b in blocks:
        va=train.billing_month.isin(b)
        tr=train.billing_month<b[0]
        yield np.where(tr)[0], np.where(va)[0]
for tr,va in folds(): print(len(tr),len(va),train.loc[va,TARGET].mean().round(3))

In [ ]:
oof={k:[] for k in candidates()}; ys=[]; fold_ap={k:[] for k in candidates()}
X=train[FEATURES]; y=train[TARGET]
for tr,va in folds():
    ys.append(y.iloc[va].to_numpy())
    for k,m in candidates().items():   # fresh pipelines: preprocessing is fit on the training fold only
        m.fit(X.iloc[tr],y.iloc[tr]); p=m.predict_proba(X.iloc[va])[:,1]
        oof[k].append(p); fold_ap[k].append(average_precision_score(y.iloc[va],p))
yo=np.concatenate(ys)
cv=pd.DataFrame({k:{'mean_AP':np.mean(v),'sd_AP':np.std(v),'pooled_AUC':roc_auc_score(yo,np.concatenate(oof[k]))} for k,v in fold_ap.items()}).T
cv.sort_values('mean_AP',ascending=False)

## 3. Choice (CV only)

In [ ]:
best=cv.mean_AP.idxmax(); print('chosen model:',best)
po=np.concatenate(oof[best])
ths=np.linspace(0.02,0.8,79); f1s=[f1_score(yo,po>=t) for t in ths]
THRESH=float(ths[int(np.argmax(f1s))])
print('chosen threshold (max pooled-OOF F1): %.2f, CV F1 %.3f'%(THRESH,max(f1s)))

**Recorded choice:** the model and threshold printed above are frozen here. The test set has not been touched.

## 4. Final fit and single test score

In [ ]:
final=candidates()[best]
final.fit(train[FEATURES],train[TARGET])
pt=final.predict_proba(test[FEATURES])[:,1]; yt=test[TARGET]
pred=pt>=THRESH
print('test rows',len(test),'accounts',test.meter_ref.nunique(),'prevalence %.3f'%yt.mean())
print('AUC %.3f  AP %.3f (prevalence baseline AP = %.3f)'%(roc_auc_score(yt,pt),average_precision_score(yt,pt),yt.mean()))
print('at threshold %.2f: precision %.3f recall %.3f F1 %.3f'%(THRESH,precision_score(yt,pred,zero_division=0),recall_score(yt,pred),f1_score(yt,pred,zero_division=0)))

## 5. Notes and caveats
- Arrears prevalence rises sharply over 2025 (about 2-4% before 2025 to ~25% by late 2025), so the base rate in the test months differs from most training data; a threshold tuned on earlier months may be miscalibrated for later ones. Interpret precision/recall with that in mind.
- `final_notice_sent` would likely boost scores but is unavailable when predicting, so it was excluded deliberately.
- Test results are a single estimate on a few months of held-out accounts; no further tuning was done after seeing them.